In [11]:
import numpy as np
import importlib

import vertex_voyage.persist as persist
importlib.reload(persist)

# logistic regression and SVM and random decision forest
from sklearn.linear_model import LogisticRegressionCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import cross_validate

from vertex_voyage.tasks.link_prediction import DatasetGenerator, heart_benchmark

class SKLearnModel:
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model
    
    def __call__(self, x,y) -> float:
        """
        Returns the probability of the positive class for the given input x and y.
        """
        print(f"Predicting for x: {x}, y: {y}")
        return self.model.predict_proba(self.preprocessor(x,y).reshape(1,-1))[0][1]


dataset = "CITESEER"
partitions = 1 
def find_best_model(dataset, partitions):
    dataset_name = dataset
    models = {
        # LogisticRegressionCV: dict(max_iter=1000),
        RandomForestClassifier: dict(n_estimators=100, random_state=42),
        # SVC: dict(probability=True, random_state=42)
    }

    preprocessors = {
        # "Hadamard": lambda u, v: u * v,
        # "Average": lambda u, v: (u + v) / 2,
        # "L1": lambda u, v: np.abs(u - v),
        # "L2": lambda u, v: (u - v) ** 2,
        "Concatenate": lambda u, v: np.concatenate((u, v))
    }
    run = persist.PersistedRun.load(f"../runs/{dataset}-partitions-lpa-{partitions}")
    dataset = run["graph"]
    removed_edges = run["removed_edges"]
    positive_edges = run["positive_edges"]
    negative_edges = run["negative_edges"]
    embedding_dict = run["embedding_dict"]
    class EM:
        def __init__(self, embedding_dict):
            self.embedding_dict = embedding_dict
        def embed_nodes(self, nodes):
            return [self.embed_node(n) for n in nodes]
        def embed_node(self, node):
            return self.embedding_dict[node]
    em = EM(embedding_dict)
    generator = DatasetGenerator(dataset, em)
    u_train: np.ndarray
    v_train: np.ndarray
    y_train: np.ndarray
    u_test: np.ndarray
    v_test: np.ndarray
    y_test: np.ndarray
    u_train, v_train, y_train, _, _, _ = generator.generate_train_val_data(0)
    u_test, v_test, y_test = generator.generate_data(positive_edges, negative_edges)
    data = []
    best_accuracy = 0
    best_model = None
    best_preprocessor = None
    to_save = {}
    if "best_sklearn_model" in run:
        to_save = run["best_sklearn_model"]
        best_model = type(to_save["model"])
        best_preprocessor = to_save["preprocessor"]
        best_accuracy = accuracy_score(y_test, to_save["model"].predict(np.array([preprocessors[to_save["preprocessor"]](u, v) for u, v in zip(u_test, v_test)])))
    else:
        for model_class, params in models.items():
            for preprocessor_name, preprocessor in preprocessors.items():
                # print(f"Evaluating model: {model_class.__name__} with preprocessor: {preprocessor_name}")
                X_train = np.array([preprocessor(u, v) for u, v in zip(u_train, v_train)])
                X_test = np.array([preprocessor(u, v) for u, v in zip(u_test, v_test)])
                y_train = np.array(y_train)
                y_test = np.array(y_test)
                model = model_class(**params)
                model.fit(X_train, y_train)
                sklearn_model = SKLearnModel(preprocessor, model)
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_model = model_class
                    best_preprocessor = preprocessor_name
                    to_save = {
                        "model": model,
                        "preprocessor": preprocessor_name
                    }
                print(f"Model: {model_class.__name__}, Preprocessor: {preprocessor_name}, Accuracy: {accuracy:.4f}, F1 Score: {f1:.4f}")
                # ranks = heart_benchmark(em, sklearn_model, dataset, positive_edges)
                # print(f"Model: {model_class.__name__}")
                # print(f"Accuracy: {accuracy:.4f}")
                # print(f"F1 Score: {f1:.4f}")
                data.append({
                    "model": model_class.__name__,
                    "preprocessor": preprocessor_name,
                    "accuracy": accuracy,
                    "f1_score": f1,
                    "dataset": dataset_name,
                    "partitions": partitions,
                    # "MRR": ranks.mrr(),
                    # "Hits@5": ranks.hits_at_k(5),
                    # "Hits@10": ranks.hits_at_k(10),
                })
        run["best_sklearn_model"] = to_save
    # Return 3 best models and preprocessors and their accuracy
    if len(data) < 3:
        return data + [{"model": None, "preprocessor": None, "accuracy": 0, "f1_score": 0, "dataset": dataset_name, "partitions": partitions}] * (3 - len(data))
    best_models = sorted(data, key=lambda x: x["accuracy"], reverse=True)[:3]
    return best_models

data = [] 
for dataset in ["CITESEER", "AstroPh", "Cit-HepPh", "Cit-HepTh", "AS-Oregon"]:
    for partitions in [1, 2, 4, 8]:
        best_models = find_best_model(dataset, partitions)
        data.append({
            "dataset": dataset,
            "partitions": partitions,
            "#1 model": best_models[0]["model"],
            "#1 preprocessor": best_models[0]["preprocessor"],
            "#1 accuracy": best_models[0]["accuracy"],
            "#1 f1_score": best_models[0]["f1_score"],
            # "#1 MRR": best_models[0]["MRR"],
            # "#1 Hits@5": best_models[0]["Hits@5"],
            # "#1 Hits@10": best_models[0]["Hits@10"],
            # "#2 model": best_models[1]["model"],
            # "#2 preprocessor": best_models[1]["preprocessor"],
            # "#2 accuracy": best_models[1]["accuracy"],
            # "#3 model": best_models[2]["model"],
            # "#3 preprocessor": best_models[2]["preprocessor"],
            # "#3 accuracy": best_models[2]["accuracy"]
        })
        print(f"Completed dataset: {dataset} with partitions: {partitions}")
        display(data[-1])

import pandas as pd
df = pd.DataFrame(data)
df


Model: RandomForestClassifier, Preprocessor: Concatenate, Accuracy: 0.7848, F1 Score: 0.7354
Completed dataset: CITESEER with partitions: 1


{'dataset': 'CITESEER',
 'partitions': 1,
 '#1 model': 'RandomForestClassifier',
 '#1 preprocessor': 'Concatenate',
 '#1 accuracy': 0.7847682119205298,
 '#1 f1_score': 0.7354138398914518}

Model: RandomForestClassifier, Preprocessor: Concatenate, Accuracy: 0.7494, F1 Score: 0.6825
Completed dataset: CITESEER with partitions: 2


{'dataset': 'CITESEER',
 'partitions': 2,
 '#1 model': 'RandomForestClassifier',
 '#1 preprocessor': 'Concatenate',
 '#1 accuracy': 0.7494481236203091,
 '#1 f1_score': 0.6825174825174826}

Model: RandomForestClassifier, Preprocessor: Concatenate, Accuracy: 0.7351, F1 Score: 0.6581
Completed dataset: CITESEER with partitions: 4


...

Model: RandomForestClassifier, Preprocessor: Concatenate, Accuracy: 0.9088, F1 Score: 0.9023
Completed dataset: AS-Oregon with partitions: 4


{'dataset': 'AS-Oregon',
 'partitions': 4,
 '#1 model': 'RandomForestClassifier',
 '#1 preprocessor': 'Concatenate',
 '#1 accuracy': 0.9088447653429603,
 '#1 f1_score': 0.9023345765989778}

Model: RandomForestClassifier, Preprocessor: Concatenate, Accuracy: 0.9145, F1 Score: 0.9092
Completed dataset: AS-Oregon with partitions: 8


{'dataset': 'AS-Oregon',
 'partitions': 8,
 '#1 model': 'RandomForestClassifier',
 '#1 preprocessor': 'Concatenate',
 '#1 accuracy': 0.9145177926766375,
 '#1 f1_score': 0.9092153909352321}

,dataset,partitions,#1 model,#1 preprocessor,#1 accuracy,#1 f1_score
0,CITESEER,1,RandomForestClassifier,Concatenate,0.784768,0.735414
1,CITESEER,2,RandomForestClassifier,Concatenate,0.749448,0.682517
2,CITESEER,4,RandomForestClassifier,Concatenate,0.735099,0.658120
3,CITESEER,8,RandomForestClassifier,Concatenate,0.728477,0.642442
4,AstroPh,1,RandomForestClassifier,Concatenate,0.953410,0.952435
5,AstroPh,2,RandomForestClassifier,Concatenate,0.950103,0.949129
6,AstroPh,4,RandomForestClassifier,Concatenate,0.948412,0.947211
7,AstroPh,8,RandomForestClassifier,Concatenate,0.946192,0.944898
8,Cit-HepPh,1,RandomForestClassifier,Concatenate,0.953910,0.953301
9,Cit-HepPh,2,RandomForestClassifier,Concatenate,0.941521,0.940099
